In [8]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith
import datetime
import json
from pathlib import Path

import pandas as pd

from datasmith.benchmark.collection import BenchmarkCollection
from datasmith.docker.context import ContextRegistry, Task
from datasmith.utils import _get_github_metadata
from scratch.notebooks.utils import merge_registries, update_cr

curr_date: str = datetime.datetime.now().isoformat()

/mnt/sdd1/atharvas/formulacode/datasmith


In [9]:
collections = [BenchmarkCollection.load(p) for p in Path("scratch/artifacts/processed").rglob("*breakpoints.fc.pkl")]
list(Path("scratch/artifacts/processed").rglob("*breakpoints.fc.pkl"))

[PosixPath('scratch/artifacts/processed/downloads/numpy/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/distributed/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/pymc3/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/joblib/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/sklearn/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/pandas/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/pandas2/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/scikit-image/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/dask/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/astropy/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/xarray/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/scipy/breakpoints.fc.pkl')]

In [10]:
results_pth = Path("scratch/artifacts/processed/downloads")
collections = [BenchmarkCollection.load(p) for p in results_pth.rglob("*breakpoints.fc.pkl")]
registries = results_pth.rglob("**/*context_registry*.json")
merged_json = merge_registries(list(registries))
registry = update_cr(ContextRegistry.deserialize(payload=json.dumps(merged_json)))
len(registry.registry)
# scratch/artifacts/processed/downloads/merged_context_registry_2025-09-09T00:38:42.145419.json : 1016 entries

registry.save_to_file(results_pth / f"merged_context_registry_{curr_date}.json")

scratch/artifacts/processed/downloads/merged_context_registry_2025-09-09T00:38:42.145419.json : 1016 entries
scratch/artifacts/processed/downloads/merged_context_registry_2025-09-12T03:42:49.965377.json : 1198 entries
scratch/artifacts/processed/downloads/merged_context_registry_2025-09-09T14:32:37.382974.json : 1156 entries
scratch/artifacts/processed/downloads/merged_context_registry_2025-09-09T01:32:38.179134.json : 1198 entries
scratch/artifacts/processed/downloads/merged_context_registry_2025-09-12T10:16:06.549234.json : 1198 entries
scratch/artifacts/processed/downloads/numpy/context_registry.json : 67 entries
scratch/artifacts/processed/downloads/distributed/context_registry.json : 102 entries
scratch/artifacts/processed/downloads/sklearn/context_registry.json : 38 entries
scratch/artifacts/processed/downloads/pandas/context_registry.json : 528 entries
scratch/artifacts/processed/downloads/pandas2/context_registry.json : 43 entries
scratch/artifacts/processed/downloads/scikit-im

10:51:01 INFO     datasmith.docker.context: Context registry saved to scratch/artifacts/processed/downloads/merged_context_registry_2025-09-12T10:50:49.251932.json


In [ ]:
bps = []
for c in collections:
    df = c.enriched_breakpoints
    df["repo_name"] = f"{c.task.owner}/{c.task.repo}"
    commits = c.commits[
        ["files_changed", "sha", "date", "file_change_summary", "message", "patch", "repo_name"]
    ].rename(columns={"sha": "gt_hash"})
    frame = (
        df.groupby(["repo_name", "hash", "gt_hash"])["delta_pct"]
        .mean()
        .reset_index()
        .rename(columns={"delta_pct": "delta_pct_mean"})
    )
    merged_frame = frame.merge(commits, on=["repo_name", "gt_hash"], how="left")
    assert all(merged_frame.notnull().all())  # noqa: S101
    bps.append(merged_frame)

all_enriched = pd.concat(bps, ignore_index=True)
useful_enriched = all_enriched[(-1 * all_enriched["delta_pct_mean"]) > 1]
print(f"Found {len(useful_enriched)} tasks with >1% mean improvement in at least one commit")
useful_enriched["repo_name"].value_counts().reset_index()

Found 362 tasks with >1% mean improvement in at least one commit


,repo_name,count
0,pandas-dev/pandas,124
1,scipy/scipy,105
2,astropy/astropy,32
3,numpy/numpy,26
4,dask/distributed,26
5,dask/dask,17
6,pymc-devs/pymc3,10
7,scikit-learn/scikit-learn,10
8,joblib/joblib,5
9,pydata/xarray,5


In [5]:
# make a csv containing:
# container_name - task.with_env("pkg").get_image_name()
# patch - git patch of the gt_sha
# message - task.get_commit_message()
# task_id - "{owner}_{repo}_{i unique to each repo}" like astropy_astropy_id1
# gt_sha - sha value of the git batch
# sha - sha value of the parent commit.
# rows = []


def get_patch(row):
    owner, repo = row["repo_name"].split("/")
    sha = row["gt_hash"]
    endpoint = f"/repos/{owner}/{repo}/commits/{sha}"
    # Ask for the commit as a unified diff (not JSON)
    diff_text = _get_github_metadata(endpoint=endpoint, params={"diff_api": "true"})["diff"]
    return diff_text


def make_task(row) -> str:
    owner, repo = row["repo_name"].split("/")
    sha = row["hash"]
    commit_date = row["date"]
    return Task(owner=owner, repo=repo, sha=sha, commit_date=commit_date).with_tag("pkg").get_image_name()


useful_enriched["container_name"] = useful_enriched.apply(make_task, axis=1)
useful_enriched["patch"] = useful_enriched.apply(get_patch, axis=1)
useful_enriched["message"] = useful_enriched["message"]
print(f"After adding container names, {len(useful_enriched)} tasks remain")
useful_enriched = useful_enriched[
    useful_enriched["container_name"].isin({t.get_image_name() for t in registry.registry})
]
print(f"After filtering for available containers, {len(useful_enriched)} tasks remain")
repo_counters = {}


def compute_task_id(row):
    repo = row["repo_name"].replace("/", "_")
    if repo not in repo_counters:
        repo_counters[repo] = 0
    repo_counters[repo] += 1
    return f"{repo}_{repo_counters[repo]}"


useful_enriched["task_id"] = useful_enriched.apply(compute_task_id, axis=1)
useful_enriched["base_commit"] = useful_enriched["hash"]

useful_enriched[["container_name", "patch", "message", "task_id", "gt_hash", "base_commit"]].to_csv(
    results_pth / f"useful_enriched.tbformat_{curr_date}.csv", index=False
)
print(f"Wrote to {results_pth.resolve() / f'useful_enriched.tbformat_{curr_date}.csv'}")

/tmp/ipykernel_2256725/3660383708.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  useful_enriched['container_name'] = useful_enriched.apply(make_task, axis=1)
/tmp/ipykernel_2256725/3660383708.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  useful_enriched['patch'] = useful_enriched.apply(get_patch, axis=1)
/tmp/ipykernel_2256725/3660383708.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See

After adding container names, 362 tasks remain
After filtering for available containers, 209 tasks remain
Wrote to /mnt/sdd1/atharvas/formulacode/datasmith/scratch/artifacts/processed/downloads/useful_enriched.tbformat_2025-09-12T10:16:06.549234.csv


In [40]:
rows = []
for _, row in useful_enriched.iterrows():
    sha = row["base_commit"]
    repo_name = row["repo_name"]
    date = row["date"]

    rows.append({
        "sha": sha,
        "repo_name": repo_name,
        "date": date,
        "kind": "commit",
        "has_asv": True,
    })

df = pd.DataFrame(rows)
out_path = Path(f"scratch/artifacts/processed/synthetic_commits_perfonly_usefulonly_{curr_date}.parquet")
# df.to_parquet(out_path, index=False)
# print(f"Wrote to {out_path.resolve()}")

In [32]:
cr = ContextRegistry.load_from_file(
    Path("scratch/artifacts/processed/downloads/merged_context_registry_2025-09-12T10:50:49.251932.json")
)
tasks = {t.get_image_name(): t for t in cr.registry}

df[~df["container_name"].isin(tasks)].shape

(0, 6)

In [42]:
all_states = {}
for _, row in df.iterrows():
    repo_name = row["repo_name"]
    sha = row["sha"]
    has_asv = row.get("has_asv", True)
    if not has_asv:
        continue
    owner, repo = repo_name.split("/")
    commit_date_unix: float = (
        0.0 if row.get("date", None) is None else datetime.datetime.fromisoformat(row["date"]).timestamp()
    )
    if (owner, repo) not in all_states:
        all_states[(owner, repo)] = [(sha, commit_date_unix)]
    else:
        all_states[(owner, repo)].append((sha, commit_date_unix))

In [47]:
all_imgs = {t.get_image_name() for t in cr.registry}
tasks = []
for (owner, repo), uniq in all_states.items():
    for sha, date in list(uniq):
        task = Task(owner, repo, sha, commit_date=float(date))
        if task.with_tag("pkg").get_image_name() in all_imgs and (sha is not None):
            tasks.append(task.with_tag("pkg"))
        else:
            print(f"main: skipping {task} not in context registry")